# Speech-to-Text Preprocessing

У цьому ноутбуці ми спочатку переглядаємо кілька прикладів із датасету, щоб побачити, як виглядають аудіо та їхні транскрипти, а потім крок за кроком готуємо дані для навчання speech-to-text моделі. Пайплайн перевіряє, чи кожному тексту відповідає реальний аудіофайл, відсіює занадто короткі або підозріло довгі записи, оцінює рівень гучності, за потреби м’яко його вирівнює, нормалізує тексти, приводить аудіо до єдиного формату й у результаті формує готові train, validation та test маніфести.

In [26]:
from __future__ import annotations

import audioop
import json
import random
import re
import statistics
import unicodedata
import wave
from collections import Counter
from dataclasses import asdict, dataclass
from pathlib import Path

DATASET_ROOT = Path('../dataset')
LABELS_PATH = DATASET_ROOT / 'labels.jsonl'
OUTPUT_ROOT = Path('../prepared_stt')

TARGET_SAMPLE_RATE = 16000
MIN_DURATION_SEC = 0.3
MAX_DURATION_SEC = 20.0
MIN_CHARS = 1
MIN_CHARS_PER_SECOND = 0.5
MAX_CHARS_PER_SECOND = 35.0
KEEP_PUNCTUATION = False

VAL_RATIO = 0.1
TEST_RATIO = 0.1
SEED = 42


In [27]:
DEFAULT_ALLOWED_CHARS = " абвгґдеєжзиіїйклмнопрстуфхцчшщьюя'"


@dataclass
class PreparedSample:
    sample_id: str
    source_path: str
    audio_filepath: str
    speaker_id: str
    text: str
    original_text: str
    duration: float
    sample_rate: int
    channels: int
    split: str


def load_labels(labels_path: Path) -> dict[str, str]:
    return json.loads(labels_path.read_text(encoding='utf-8'))


def normalize_text(text: str, keep_punctuation: bool = False) -> str:
    text = unicodedata.normalize('NFKC', text)
    text = text.replace('\u0301', '')
    text = text.replace('’', "'").replace('`', "'").replace('ʼ', "'")
    text = text.replace('«', '"').replace('»', '"')
    text = text.replace('…', ' ')
    text = text.lower().strip()

    if keep_punctuation:
        return re.sub(r'\s+', ' ', text)

    allowed = set(DEFAULT_ALLOWED_CHARS)
    normalized_chars = [ch if ch in allowed else ' ' for ch in text]
    text = ''.join(normalized_chars)
    text = re.sub(r'\s+', ' ', text).strip()
    text = re.sub(r"\s+'\s+", "'", text)
    return text


def get_speaker_id(rel_path: str) -> str:
    return Path(rel_path).parent.name


def iter_existing_records(dataset_root: Path, labels: dict[str, str]):
    for rel_path, text in labels.items():
        wav_path = dataset_root.parent / rel_path
        if wav_path.exists():
            yield rel_path, wav_path, text


def assign_splits(speaker_ids: set[str], val_ratio: float, test_ratio: float, seed: int) -> dict[str, str]:
    speakers = sorted(speaker_ids)
    rng = random.Random(seed)
    rng.shuffle(speakers)

    total = len(speakers)
    test_count = round(total * test_ratio)
    val_count = round(total * val_ratio)

    if total >= 3:
        test_count = max(1, min(test_count, total - 2))
        val_count = max(1, min(val_count, total - test_count - 1))

    split_map = {}
    for idx, speaker in enumerate(speakers):
        if idx < test_count:
            split_map[speaker] = 'test'
        elif idx < test_count + val_count:
            split_map[speaker] = 'validation'
        else:
            split_map[speaker] = 'train'
    return split_map


def convert_wav(src_path: Path, dst_path: Path, target_sample_rate: int) -> tuple[float, int, int]:
    with wave.open(str(src_path), 'rb') as src_wav:
        channels = src_wav.getnchannels()
        sample_width = src_wav.getsampwidth()
        source_rate = src_wav.getframerate()
        frames = src_wav.getnframes()
        audio = src_wav.readframes(frames)

    if channels > 1:
        audio = audioop.tomono(audio, sample_width, 0.5, 0.5)
        channels = 1

    if sample_width != 2:
        audio = audioop.lin2lin(audio, sample_width, 2)
        sample_width = 2

    if source_rate != target_sample_rate:
        audio, _ = audioop.ratecv(audio, sample_width, channels, source_rate, target_sample_rate, None)

    frame_count = len(audio) // (sample_width * channels)
    duration = frame_count / target_sample_rate

    dst_path.parent.mkdir(parents=True, exist_ok=True)
    with wave.open(str(dst_path), 'wb') as dst_wav:
        dst_wav.setnchannels(channels)
        dst_wav.setsampwidth(sample_width)
        dst_wav.setframerate(target_sample_rate)
        dst_wav.writeframes(audio)

    return duration, target_sample_rate, channels


def write_jsonl(path: Path, rows: list[dict]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open('w', encoding='utf-8') as handle:
        for row in rows:
            handle.write(json.dumps(row, ensure_ascii=False) + '\n')


In [28]:
def inspect_dataset(dataset_root: Path, labels_path: Path) -> dict:
    labels = load_labels(labels_path)
    existing_records = list(iter_existing_records(dataset_root, labels))

    durations = []
    sample_rates = Counter()
    channels = Counter()
    empty_existing = 0

    for _, wav_path, text in existing_records:
        with wave.open(str(wav_path), 'rb') as wav_file:
            durations.append(wav_file.getnframes() / wav_file.getframerate())
            sample_rates[wav_file.getframerate()] += 1
            channels[wav_file.getnchannels()] += 1
        if not text.strip():
            empty_existing += 1

    return {
        'label_entries': len(labels),
        'existing_audio_with_labels': len(existing_records),
        'missing_audio_for_label': len(labels) - len(existing_records),
        'empty_transcripts_existing_audio': empty_existing,
        'sample_rates': dict(sample_rates),
        'channels': dict(channels),
        'min_duration_sec': min(durations),
        'mean_duration_sec': statistics.mean(durations),
        'max_duration_sec': max(durations),
    }


dataset_stats = inspect_dataset(DATASET_ROOT, LABELS_PATH)
dataset_stats


{'label_entries': 29232,
 'existing_audio_with_labels': 18303,
 'missing_audio_for_label': 10929,
 'empty_transcripts_existing_audio': 104,
 'sample_rates': {44100: 18303},
 'channels': {1: 18303},
 'min_duration_sec': 0.14,
 'mean_duration_sec': 5.957715179462668,
 'max_duration_sec': 17.28}

In [29]:
def prepare_stt_dataset(
    dataset_root: Path,
    labels_path: Path,
    output_root: Path,
    target_sample_rate: int = 16000,
    min_duration_sec: float = 0.3,
    max_duration_sec: float = 20.0,
    min_chars: int = 1,
    min_chars_per_second: float = 0.5,
    max_chars_per_second: float = 35.0,
    keep_punctuation: bool = False,
    val_ratio: float = 0.1,
    test_ratio: float = 0.1,
    seed: int = 42,
) -> dict:
    labels = load_labels(labels_path)
    source_records = list(iter_existing_records(dataset_root, labels))

    speaker_ids = {get_speaker_id(rel_path) for rel_path, _, _ in source_records}
    split_by_speaker = assign_splits(speaker_ids, val_ratio, test_ratio, seed)

    prepared_samples = []
    dropped = {
        'missing_audio': len(labels) - len(source_records),
        'empty_transcript': 0,
        'normalized_to_empty': 0,
        'duration_out_of_range': 0,
        'invalid_char_rate': 0,
    }

    for rel_path, source_path, original_text in source_records:
        if not original_text.strip():
            dropped['empty_transcript'] += 1
            continue

        normalized_text = normalize_text(original_text, keep_punctuation=keep_punctuation)
        if len(normalized_text) < min_chars:
            dropped['normalized_to_empty'] += 1
            continue

        speaker_id = get_speaker_id(rel_path)
        split = split_by_speaker[speaker_id]
        sample_id = Path(rel_path).with_suffix('').as_posix().replace('/', '__')
        prepared_audio_path = output_root / 'audio' / split / f'{sample_id}.wav'

        duration, sample_rate, channels = convert_wav(source_path, prepared_audio_path, target_sample_rate)

        if not (min_duration_sec <= duration <= max_duration_sec):
            dropped['duration_out_of_range'] += 1
            prepared_audio_path.unlink(missing_ok=True)
            continue

        chars_per_second = len(normalized_text.replace(' ', '')) / duration
        if not (min_chars_per_second <= chars_per_second <= max_chars_per_second):
            dropped['invalid_char_rate'] += 1
            prepared_audio_path.unlink(missing_ok=True)
            continue

        prepared_samples.append(
            PreparedSample(
                sample_id=sample_id,
                source_path=str(source_path.resolve()),
                audio_filepath=str(prepared_audio_path.resolve()),
                speaker_id=speaker_id,
                text=normalized_text,
                original_text=original_text,
                duration=round(duration, 6),
                sample_rate=sample_rate,
                channels=channels,
                split=split,
            )
        )

    manifests_dir = output_root / 'manifests'
    reports_dir = output_root / 'reports'

    for split in ('train', 'validation', 'test'):
        split_rows = [asdict(sample) for sample in prepared_samples if sample.split == split]
        write_jsonl(manifests_dir / f'{split}.jsonl', split_rows)

    vocabulary = sorted({ch for sample in prepared_samples for ch in sample.text if ch != ' '})

    summary = {
        'dataset_root': str(dataset_root.resolve()),
        'output_root': str(output_root.resolve()),
        'target_sample_rate': target_sample_rate,
        'keep_punctuation': keep_punctuation,
        'prepared_samples': len(prepared_samples),
        'dropped': dropped,
        'splits': {
            split: sum(1 for sample in prepared_samples if sample.split == split)
            for split in ('train', 'validation', 'test')
        },
        'speaker_counts': {
            split: len({sample.speaker_id for sample in prepared_samples if sample.split == split})
            for split in ('train', 'validation', 'test')
        },
        'duration_hours': round(sum(sample.duration for sample in prepared_samples) / 3600, 3),
        'vocabulary_size': len(vocabulary),
        'vocabulary': vocabulary,
    }

    reports_dir.mkdir(parents=True, exist_ok=True)
    (reports_dir / 'summary.json').write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding='utf-8')
    (reports_dir / 'vocabulary.txt').write_text(''.join(vocabulary), encoding='utf-8')

    return {
        'summary': summary,
        'prepared_samples': prepared_samples,
    }


In [30]:
result = prepare_stt_dataset(
    dataset_root=DATASET_ROOT,
    labels_path=LABELS_PATH,
    output_root=OUTPUT_ROOT,
    target_sample_rate=TARGET_SAMPLE_RATE,
    min_duration_sec=MIN_DURATION_SEC,
    max_duration_sec=MAX_DURATION_SEC,
    min_chars=MIN_CHARS,
    min_chars_per_second=MIN_CHARS_PER_SECOND,
    max_chars_per_second=MAX_CHARS_PER_SECOND,
    keep_punctuation=KEEP_PUNCTUATION,
    val_ratio=VAL_RATIO,
    test_ratio=TEST_RATIO,
    seed=SEED,
)

result['summary']


{'dataset_root': '/Users/khrystynakorets/Desktop/hw3_sound-processing/dataset',
 'output_root': '/Users/khrystynakorets/Desktop/hw3_sound-processing/prepared_stt',
 'target_sample_rate': 16000,
 'keep_punctuation': False,
 'prepared_samples': 18145,
 'dropped': {'missing_audio': 10929,
  'empty_transcript': 104,
  'normalized_to_empty': 35,
  'duration_out_of_range': 1,
  'invalid_char_rate': 18},
 'splits': {'train': 14530, 'validation': 1613, 'test': 2002},
 'speaker_counts': {'train': 57, 'validation': 7, 'test': 7},
 'duration_hours': 30.032,
 'vocabulary_size': 34,
 'vocabulary': ["'",
  'а',
  'б',
  'в',
  'г',
  'д',
  'е',
  'ж',
  'з',
  'и',
  'й',
  'к',
  'л',
  'м',
  'н',
  'о',
  'п',
  'р',
  'с',
  'т',
  'у',
  'ф',
  'х',
  'ц',
  'ч',
  'ш',
  'щ',
  'ь',
  'ю',
  'я',
  'є',
  'і',
  'ї',
  'ґ']}

In [32]:
sample_preview = [asdict(sample) for sample in result['prepared_samples'][:5]]
sample_preview


[{'sample_id': 'dataset__toronto_157__toronto_157_0',
  'source_path': '/Users/khrystynakorets/Desktop/hw3_sound-processing/dataset/toronto_157/toronto_157_0.wav',
  'audio_filepath': '/Users/khrystynakorets/Desktop/hw3_sound-processing/prepared_stt/audio/test/dataset__toronto_157__toronto_157_0.wav',
  'speaker_id': 'toronto_157',
  'text': 'слава ісу ви сі дивите програму грати песик дужка гривня знак питання долар нуль я є її ведучий майкл щур вйо до новин',
  'original_text': 'Слава Ісу! Ви сі дивите програму «Грати, песик, дужка, гривня, знак питання, долар, нуль» ₴?$0»). Я є її ведучий, Майкл Щур. Вйо до новин!',
  'duration': 7.06,
  'sample_rate': 16000,
  'channels': 1,
  'split': 'test'},
 {'sample_id': 'dataset__toronto_157__toronto_157_1',
  'source_path': '/Users/khrystynakorets/Desktop/hw3_sound-processing/dataset/toronto_157/toronto_157_1.wav',
  'audio_filepath': '/Users/khrystynakorets/Desktop/hw3_sound-processing/prepared_stt/audio/test/dataset__toronto_157__toronto_1